# kwargs-pass-through-recipe — worked example 2: keepdims kwarg stored and passed to backward — sum_back example

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `kwargs-pass-through-recipe`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

A `sum` with `keepdims=True` produces an output with the same number of dimensions as the input. The backward function must know this to broadcast the gradient correctly. If `keepdims` is stored faithfully in the Recipe, the backward can call `expand_as` or similar; if it is lost, the gradient shape will be wrong.

## Worked solution

Step 1: Wrap `np.sum` with `wrap_forward_fn`. The wrapper stores whatever kwargs were passed into the Recipe — including `keepdims`.

Step 2: Call `wrapped_sum(x, axis=0, keepdims=True)` on a 2D input. The output should have shape `(1, 4)` (dimension preserved) rather than `(4,)`.

Step 3: Verify `out.recipe.kwargs == {'axis': 0, 'keepdims': True}`.

Step 4: Implement a simple `sum_back(grad_out, out, x, axis=None, keepdims=False)` that broadcasts `grad_out` back to the input shape using `np.broadcast_to`. Show that it uses the stored kwargs correctly by splatting `**recipe.kwargs` into it.

In [ ]:
import numpy as np
from dataclasses import dataclass
from typing import Any, Callable, Optional

@dataclass
class Recipe:
    func: Any
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array):
        self.array = array
        self.recipe: Optional[Recipe] = None

def wrap_forward_fn(fwd_fn: Callable) -> Callable:
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_raw = fwd_fn(*raw_args, **kwargs)
        parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
        out = MiniTensor(out_raw)
        out.recipe = Recipe(fwd_fn, raw_args, dict(kwargs), parents)
        return out
    return tensor_func

def sum_back(grad_out, out, x, axis=None, keepdims=False):
    """Backward for np.sum: broadcast grad back to input shape."""
    if not keepdims and axis is not None:
        # Restore the reduced dimension so broadcast_to works
        grad_out = np.expand_dims(grad_out, axis=axis)
    return np.broadcast_to(grad_out, x.shape).copy()

wrapped_sum = wrap_forward_fn(np.sum)

x = MiniTensor(np.ones((3, 4)))

# Forward with keepdims=True
out_k = wrapped_sum(x, axis=0, keepdims=True)
print('forward shape (keepdims=True):', out_k.array.shape)   # (1, 4)
print('recipe kwargs:', out_k.recipe.kwargs)                  # {'axis': 0, 'keepdims': True}
assert out_k.recipe.kwargs == {'axis': 0, 'keepdims': True}

# Backward via recipe replay
grad_out = np.ones_like(out_k.array)
parent_x = out_k.recipe.parents[0]
grad_x = sum_back(grad_out, out_k.array, parent_x.array, **out_k.recipe.kwargs)
print('backward grad shape:', grad_x.shape)  # (3, 4)
assert grad_x.shape == (3, 4)
print('All checks passed.')